# Freeze and reload a reference

A reference is written once, checksummed, and never redefined by a query. The frozen format
is a directory (or zip) of NumPy arrays, CSV and JSON files: no pickle, readable without
StateTrace.

In [1]:
import tempfile
import warnings
from pathlib import Path

import pandas as pd

import statetrace as st
from statetrace.datasets import simulate_transport

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)
workdir = Path(tempfile.mkdtemp(prefix="statetrace_tutorial_"))
# show warnings as one line, without the source file
warnings.formatwarning = (
    lambda message, category, *args, **kwargs: f"{category.__name__}: {message}\n"
)
st.__version__

'0.1.0'

In [2]:
ref_adata, query_adata, truth = simulate_transport(
    n_ref_groups=8,
    n_query_groups=6,
    n_states=5,
    n_latent=10,
    n_genes=300,
    shifted_states={"S1": 9.0},
    absent_states=["S4"],
    minority_states={"S3": 0.25},
    seed=5,
)
ref_adata

AnnData object with n_obs × n_vars = 1600 × 300
    obs: 'donor_id', 'cell_state', 'cell_type', 'batch', 'study'
    uns: 'statetrace_synthetic'
    obsm: 'X_latent'
    layers: None (.X)

In [3]:
ref = st.Reference(
    ref_adata, group_key="donor_id", state_key="cell_state", latent_key="X_latent"
).fit()
ref.leave_one_group_out()
path = ref.freeze(workdir / "reference.statetrace")
path.name

'reference.statetrace'

## What is inside

`manifest.json` records the format version, the keys, the backend, the thresholds and a
checksum of every file. `validation/` holds the donor holdout, and `expression/` the frozen
expression space.

In [4]:
sorted(str(p.relative_to(path)) for p in path.rglob("*") if p.is_file())

['categories.json',
 'cells.npz',
 'centroids.npy',
 'expression/cell_center.npy',
 'expression/cell_components.npy',
 'expression/cell_coords.npy',
 'expression/cell_rows.csv',
 'expression/cell_scale.npy',
 'expression/components.npy',
 'expression/genes.json',
 'expression/mu.npy',
 'expression/n_groups.npy',
 'expression/null.csv',
 'expression/null_profiles.npy',
 'expression/null_stats.csv',
 'expression/params.json',
 'expression/profiles.npy',
 'expression/rows.csv',
 'expression/sigma.npy',
 'geometry.npy',
 'latent.npy',
 'manifest.json',
 'state_table.csv',
 'validation/cell_predictions.csv.gz',
 'validation/config.json',
 'validation/confusion_matrix.csv',
 'validation/distance_calibration.csv',
 'validation/group_metrics.csv',
 'validation/group_summary.csv',
 'validation/holdout_evidence.csv.gz',
 'validation/overall.json',
 'validation/state_metrics.csv']

In [5]:
import json

manifest = json.loads((path / "manifest.json").read_text())
{k: manifest[k] for k in ("format_version", "keys", "backend")}

{'format_version': 4,
 'keys': {'group': 'donor_id',
  'state': 'cell_state',
  'celltype': None,
  'study': None,
  'latent': 'X_latent'},
 'backend': {'name': 'knn',
  'params': {'n_neighbors': 15,
   'metric': 'cosine',
   'weights': 'uniform',
   'n_jobs': None}}}

Because the files are plain, a collaborator can read the state table with pandas alone:

In [6]:
pd.read_csv(path / "state_table.csv", index_col=0)

,n_cells,n_groups,n_studies,eligible,reason
state,,,,,
S0,344,8,NaN,True,NaN
S1,379,8,NaN,True,NaN
S2,389,8,NaN,True,NaN
S3,86,2,NaN,False,"few_groups,few_cells"
S4,402,8,NaN,True,NaN


## Reload

`Reference.load` verifies every checksum and restores the reference; `fit()` rebuilds the kNN
index from the stored latent.

In [7]:
ref2 = st.Reference.load(path).fit()
ref2

<StateReference 1600 cells, 8 groups, 5 states, 10-d latent, expression, frozen, validated>

A modified file is refused:

In [8]:
import shutil

tampered = workdir / "tampered.statetrace"
shutil.copytree(path, tampered)
table = tampered / "state_table.csv"
table.write_text(table.read_text().replace("True", "False", 1))
try:
    st.Reference.load(tampered)
except st.FrozenReferenceError as err:
    print(err)

/var/tmp/statetrace_tutorial_rvk75j70/tampered.statetrace: checksum mismatch for state_table.csv
